# Part 3 — The API in Depth (Human)

*Written for idtrack 0.1.2.*

A hands-on tour of idtrack's public API with the **human** graph.

**Learning objectives**
- Load a graph into `idtrack.API`.
- Convert single identifiers across releases and into external databases.
- Read a result: its candidates, the best answer, ties, statuses and reasons.
- Convert batches and summarize the outcomes (1→0 / 1→1 / 1→n).
- Ask for the explained path of an answer.
- Know the options (bridges, search budget) and the questions you can ask the graph (databases, assemblies, releases, names).

> **Prerequisite:** Part 2 (`02_initialization_graph.ipynb`), so that the graph loads from your local repository. The concepts behind every answer (history travel, bridges, ranking, ties, statuses): [How idtrack answers](../concepts.rst); how a string is read: [How an identifier is read](../reading.rst).

## 3.1 — The API

`idtrack.API` is the entry point. It
- builds or loads one graph (`API.build`, `API.load`): one organism, up to its snapshot release;
- converts: `convert(...)` and `convert_many(...)` return `idtrack.results.ConversionResult` objects;
- answers questions about the graph (`describe()`, `releases()`, `external_databases()`, `genes_named(...)`, ...).

The cell below loads the human graph and defines a small helper that prints the main fields of a result.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import idtrack
from idtrack import API

# The local repository (IDTRACK_LOCAL_REPO is a convention of these tutorials; idtrack reads no such variable).
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

API.configure_logger()

organism = idtrack.organisms.resolve_organism('human')  # offline; API.ensembl_organism also asks Ensembl's newest release
SNAPSHOT_RELEASE = 115  # fixed, so that the answers stay the same

api = API.build(organism, SNAPSHOT_RELEASE, str(LOCAL_REPOSITORY))
print('Ready:', organism, 'snapshot release', SNAPSHOT_RELEASE)


def show(result, strategy='best'):
    """Print the main fields of a conversion result and its candidates of one view ('best' or 'all')."""
    print(f'{result.query_id!r} read as {result.graph_id!r} (match: {result.match}) at release {result.from_release} '
          f'-> {result.final_database} at {result.to_release}: status={result.status}, tie={result.tie}, '
          f'decided_by={result.decided_by}')
    for c in result.view(strategy):
        print(f'  {c.gene} -> {c.target}  best={c.best} entry={c.entry_gene} bridges={c.bridges} score={c.score:.3f}')
    if result.alternative_locus_genes:
        print('  copies on alternative loci:', list(result.alternative_locus_genes))
    if result.reason:
        print('  reason:', API.reason_text(result))

2026-11-24 11:52:03 WARNING:idtrack.build.externals: No externals YAML in the local repository (`idtrack_cache/homo_sapiens_externals_modified.yml`); the package uses its default file for homo_sapiens (`.../idtrack/default_config/homo_sapiens_externals_modified.yml`).


Ready: homo_sapiens snapshot release 115


## 3.2 — Converting one identifier

In [2]:
show(api.convert('TP53', to_release=SNAPSHOT_RELEASE))

'TP53' read as 'TP53' (match: exact) at release 115 -> ensembl_gene at 115: status=converted, tie=False, decided_by=None
  ENSG00000141510.20 -> ENSG00000141510.20  best=True entry=ENSG00000141510.20 bridges=0 score=1.000
  copies on alternative loci: ['LRG_321.1']


`graph_id` is the identifier of the graph the string was read as, `match` how it matched (here exactly), and `from_release` the release at which it was read: without `from_release`, an identifier is read at the target release if it exists there, otherwise at the latest release at which it exists. idtrack also tries letter-case and separator variants (`tp53`; `HLA_A` for `HLA-A`) and says so in `match`. If nothing matches, the status is `not_found`.

`alternative_locus_genes` lists genes the identifier also leads to that are copies on alternative loci, assembly patches or LRG records (`LRG_321.1` for TP53): they are reported apart from the answer, which is the gene of the reference sequence.

### 3.2.1 History travel: convert to an older release

Published datasets often use older releases. The same gene at release 105:

In [3]:
older_release = SNAPSHOT_RELEASE - 10
show(api.convert('TP53', to_release=older_release))

'TP53' read as 'TP53' (match: exact) at release 105 -> ensembl_gene at 105: status=converted, tie=False, decided_by=None
  ENSG00000141510.18 -> ENSG00000141510.18  best=True entry=ENSG00000141510.18 bridges=0 score=1.000
  copies on alternative loci: ['LRG_321.1']


### 3.2.2 Convert into an external database (HGNC)

To get the gene's id in another database, pass `final_database=...`:

In [4]:
show(api.convert('ENSG00000141510', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol'))

'ENSG00000141510' read as 'ENSG00000141510' (match: exact) at release 115 -> HGNC Symbol at 115: status=converted, tie=False, decided_by=None
  ENSG00000141510.20 -> TP53  best=True entry=ENSG00000141510.20 bridges=0 score=1.000


### 3.2.3 Which databases can I name?

Two lists, which differ on purpose:

- `external_databases()`: every name of a database under which the graph holds external ids, at any release (the valid `from_database`). It is longer than the selection of the externals configuration, for two reasons. A selected database is, at every release, every database of Ensembl whose links continue it (RefSeq's mRNA accessions are filed as `RefSeq_dna` up to release 63 and as `RefSeq_mRNA` from 64; the graph lists them as `RefSeq_mRNA`). And the name Ensembl displays for a gene at a release is in the graph under the database Ensembl filed it under, selected or not (here `HGNC (curated)`, `IMGT/GENE-DB` and others).
- `final_databases(to_release)`: the values `final_database` accepts at a target release: the databases of the assembly that serves that release, plus `ensembl_gene` (with version) and `base_ensembl_gene` (without).

Pass the name as `final_databases` lists it (letter case does not matter). The filing of each database: [External databases](../databases.rst).

In [5]:
names = api.external_databases()
print(len(names), 'names in the graph, e.g.', names[:8])
print('final_database at', SNAPSHOT_RELEASE, ':', api.final_databases(SNAPSHOT_RELEASE)[:12], '...')

35 names in the graph, e.g. ['CCDS', 'CCDS label', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC (automatic)', 'HGNC (curated)', 'HGNC Symbol']
final_database at 115 : ['CCDS', 'CCDS label', 'Clone_based_ensembl_gene', 'Clone_based_vega_gene', 'EntrezGene', 'HGNC Symbol', 'Havana gene', 'Havana transcript', 'Havana translation', 'NCBI gene (formerly Entrezgene)', 'RFAM', 'RFAM label'] ...


### 3.2.4 Reading a result

A result is an `idtrack.results.ConversionResult`, a frozen dataclass (its docstring lists every field):
- `query_id`: exactly what you gave; `graph_id`: the identifier of the graph it was read as; `match`: how it matched (`exact`, `case`, `separator`, `version`, `version_newer_than_graph`, with prefixes such as `synonym:`);
- `from_release`: the release it was read at; `from_release_inferred`: chosen by idtrack (no `from_release` given); `from_release_fallback`: the `from_release` you gave does not hold the identifier, so it was read where the graph holds it;
- `candidates`: every target the search reached, ranked, the best marked (`candidate.best`); `targets('best')` and `targets('all')` are the target ids of the two views `view('best')` and `view('all')`;
- `tie`: the best answer is shared by several genes or targets that are equal in all evidence; `decided_by`: the criterion that put the best answer before the runner-up;
- `alternative_locus_genes`: copies on alternative loci and patches, reported apart from the answer;
- `status`: `converted`, `tie`, `no_target`, `not_found`, `unknown_version` (`known_versions` lists the versions that exist), `no_conversion` or `error`; `reason` (the first of `reasons`) names why a result has no target in the database asked for, and `API.reason_text(result)` says it in a sentence;
- `no_target`: genes were reached, but none has an id in the requested database (the candidates are then the genes); `no_target_genes`: genes without such an id when others have one.

There can be 0, 1 or many targets, because ambiguity is real and common. Every result is strict JSON with `result.to_dict()`, and `ConversionResult.from_dict(...)` reads it back. Every status and reason, with what to do: [How idtrack answers](../concepts.rst).

### 3.2.5 Versions, spelling variants and releases

A few identifiers that show how idtrack reads what you give it:
- an Ensembl id with a version that never existed is not answered with another version's gene (`known_versions` names the versions that exist), except a version higher than all the graph holds of an id still alive at the snapshot release, which may come from a newer release and is read as the latest version;
- a `from_release` at which the graph does not hold the identifier is set aside and reported (`from_release_fallback`), while a release that is not a release of the graph at all is an error that names the valid ones.

In [6]:
from idtrack.errors import InputError

examples = [
    ('ENSG00000141510.5', {}),                      # an old version: read where it exists (release 54), followed to 115
    ('ENSG00000141510.3', {}),                      # a version the graph does not hold (its first is .4): nothing
    ('ENSG00000141510.25', {}),                     # newer than the graph's latest version: read as the latest version
    ('hla_a', {}),                                  # letter case and separator variant of HLA-A
    ('NM_000546.9', {}),                            # a RefSeq accession with a version newer than the graph's
    ('TP53', {'from_release': 50, 'to_release': 50}),  # read and converted at an NCBI36 release
    ('ENSG00000141510.20', {'from_release': 100}),  # this version did not exist at 100: read where it exists, flagged
    ('ENSG00000141510', {'from_release': 40}),      # not a release of the graph: an input error names the valid ones
]
for query, kwargs in examples:
    try:
        r = api.convert(query, **kwargs)
    except InputError as error:
        print(f'{query!r} {kwargs} -> {type(error).__name__}: {error}')
        continue
    versions = r.known_versions
    known = f' known_versions: {len(versions)}, {versions[0]} to {versions[-1]}' if versions else ''
    print(f"{query!r} {kwargs} -> {list(r.targets())} | graph_id={r.graph_id!r} match={r.match!r} "
          f"from_release={r.from_release} fallback={r.from_release_fallback} status={r.status}{known}")

'ENSG00000141510.5' {} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.5' match='exact' from_release=54 fallback=False status=converted


'ENSG00000141510.3' {} -> [] | graph_id=None match=None from_release=None fallback=False status=unknown_version known_versions: 17, ENSG00000141510.4 to ENSG00000141510.20
'ENSG00000141510.25' {} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.20' match='version_newer_than_graph' from_release=115 fallback=False status=converted


'hla_a' {} -> ['ENSG00000206503.16'] | graph_id='HLA-A' match='separator' from_release=115 fallback=False status=converted


'NM_000546.9' {} -> ['ENSG00000141510.20'] | graph_id='NM_000546' match='version_newer_than_graph' from_release=115 fallback=False status=converted
'TP53' {'from_release': 50, 'to_release': 50} -> ['ENSG00000141510.5'] | graph_id='TP53' match='exact' from_release=50 fallback=False status=converted
'ENSG00000141510.20' {'from_release': 100} -> ['ENSG00000141510.20'] | graph_id='ENSG00000141510.20' match='exact' from_release=115 fallback=True status=converted
'ENSG00000141510' {'from_release': 40} -> UnknownReleaseError: `from_release`=40 is not a release of this graph, which covers the Ensembl releases 48 to 115.


### 3.2.6 Several candidates: the best answer, the others, and ties

One conversion returns every candidate the search reached, ranked, the best marked:
- `result.view('best')`: the best answer: one target, or, when several genes or targets are equal in all evidence, all of them with `tie=True`;
- `result.view('all')`: every target found, best first.

Candidates are ranked by these criteria, in this order (`idtrack.results.CRITERIA`): `own_version` (a queried Ensembl id that exists at the target release is its own answer), `final_confidence`, `bridges` (fewer first), `bridge_gap`, `display_name` (the string is Ensembl's display name of the gene it entered), `entry_support` (how many databases attach the string to that gene), `score` (Ensembl's mapping score, between the successors of one gene), `same_as_query` (the target is the string itself), and `annotation` (the richness of annotation, as the last resort). `decided_by` names the first criterion on which the best answer and the runner-up differ. What each measures: [How idtrack answers](../concepts.rst).

`HBA1` is a symbol that also reaches HBA2's gene; the display name decides:

In [7]:
result = api.convert('HBA1', to_release=SNAPSHOT_RELEASE)
show(result, 'all')
# The ranking record of the best candidate
result.view('best')[0]

'HBA1' read as 'HBA1' (match: exact) at release 115 -> ensembl_gene at 115: status=converted, tie=False, decided_by=display_name
  ENSG00000206172.9 -> ENSG00000206172.9  best=True entry=ENSG00000206172.9 bridges=0 score=1.000
  ENSG00000188536.14 -> ENSG00000188536.14  best=False entry=ENSG00000188536.14 bridges=0 score=1.000
  copies on alternative loci: ['LRG_1225.1', 'LRG_1240.1']


Candidate(gene='ENSG00000206172.9', target='ENSG00000206172.9', best=True, entry_gene='ENSG00000206172.9', entry_databases=('HGNC Symbol', 'NCBI gene (formerly Entrezgene)', 'UniProtKB Gene Name'), authority=True, display_name=True, display_rank=0, entry_support=3, own_version=False, final_confidence=0, bridges=0, bridge_gap=0, score=1.0, unscored_steps=0, same_as_query=False, external_steps=1, history_steps=0, annotation=None, path=(), final_path=())

Where the candidates are equal in all evidence, idtrack returns them all as a **tie** instead of picking one by position. NCBI's gene id `132989` is attached by Ensembl to two genes at release 115:

In [8]:
show(api.convert('132989', to_release=SNAPSHOT_RELEASE))

'132989' read as '132989' (match: exact) at release 115 -> ensembl_gene at 115: status=tie, tie=True, decided_by=None
  ENSG00000163633.13 -> ENSG00000163633.13  best=True entry=ENSG00000163633.13 bridges=0 score=1.000
  ENSG00000285458.1 -> ENSG00000285458.1  best=True entry=ENSG00000285458.1 bridges=0 score=1.000


## 3.3 — Batch conversion

Most work starts from a list (the genes of a count matrix, markers, hits). idtrack provides:
- `convert_many(...)`: one result per identifier, with the same arguments; an error of one identifier never stops the batch;
- `API.classify(...)` and `API.classification_summary(...)`, to count the outcomes.

In [9]:
genes = ['TP53', 'BRCA1', 'BRCA2', 'BRAF', 'KRAS', 'NOT_A_REAL_GENE']
results = api.convert_many(genes, to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol')
for r in results:
    print(f'{r.query_id:<16} {r.status:<10} {list(r.targets())}')

TP53             converted  ['TP53']
BRCA1            converted  ['BRCA1']
BRCA2            converted  ['BRCA2']
BRAF             converted  ['BRAF']
KRAS             converted  ['KRAS']
NOT_A_REAL_GENE  not_found  []


In [10]:
summary = API.classify(results)  # the bins of the best view; API.classify(results, 'all') for every target
{k: len(v) for k, v in summary.items()}

{'changed_only_1_to_n': 0,
 'changed_only_1_to_1': 0,
 'alternative_target_1_to_1': 0,
 'alternative_target_1_to_n': 0,
 'matching_1_to_0': 1,
 'matching_1_to_1': 5,
 'matching_1_to_n': 0,
 'read_at_another_release': 0,
 'input_identifiers': 6}

The summary in words:

In [11]:
print(API.classification_summary(summary))


IDTrack conversion summary:
  Total processed: 6
  1→0: 1 (16.7%)
  Read at another release: 0 (0.0%)
  1→1: 5 (83.3%)
    Changed only: 0 (0.0%)
    Alternative targets: 0 (0.0%)
    Rest: 5 (100.0%)
  1→n: 0 (0.0%)
    Changed only: 0 (0.0%)
    Alternative targets: 0 (0.0%)
  Statuses:
    converted: 5
    not_found: 1


How to read the bins:
- **1→0**: no candidate (not found, no conversion, or an error).
- **Read at another release**: an answer with candidates for an identifier that the graph does not hold at the `from_release` given; it answers what the identifier meant where it was read.
- **1→1** and **1→n**: one target, or several, in the view counted (`'best'` here).
- **Changed only**: the target is not the identifier read (compared without version and letter case): `TP53` converted to `HGNC Symbol` is unchanged, `TP53` converted to `ensembl_gene` is changed.
- **Alternative targets**: `no_target` results, whose targets are Ensembl genes standing in for a missing id of the requested database.
- **Rest**: the 1→1 answers that are their own identifier.

## 3.4 — The explained path

With `explain=True` the result holds `path`, an `idtrack.results.ExplainedPath`: one graph per conversion with every node (its kind, label, databases, the releases at which a gene is active and the assemblies that serve them, the releases at which it lies outside the reference sequence) and every step once. A step (`idtrack.results.PathStep`) has its kind (`entry` from the identifier to the gene it stands for, `history` for an event of Ensembl's history, `self_loop`, `bridge_near` and `bridge_far` for the two sides of a bridge, `final` from the gene to its id in the final database, `identity` for a gene that is its own answer), the releases, Ensembl's mapping score, the splits and merges of a history event, the databases and the assembly of a cross-reference, and the reason it was taken. Each candidate lists its steps by index: `path` from the identifier to the gene, `final_path` from the gene to the target.

In [12]:
explained = api.convert('TP53', to_release=SNAPSHOT_RELEASE, final_database='HGNC Symbol', explain=True)
best = explained.view('best')[0]
for index in best.path + best.final_path:
    step = explained.path.steps[index]
    print(f'{step.kind:<8} {step.source} -> {step.target} | release {step.release} | {", ".join(step.databases)}')

entry    TP53 -> ENSG00000141510.20 | release 115 | HGNC Symbol, NCBI gene (formerly Entrezgene), UniProtKB Gene Name
final    ENSG00000141510.20 -> TP53 | release 115 | HGNC Symbol, NCBI gene (formerly Entrezgene), UniProtKB Gene Name


In [13]:
# The nodes of the explanation: kind, label, active releases (None: still active at the snapshot release), databases
for node in explained.path.nodes:
    print(node.kind, node.label, node.active, node.databases[:3])

external TP53 () ('HGNC Symbol', 'NCBI gene (formerly Entrezgene)', 'UniProtKB Gene Name')
ensembl_gene ENSG00000141510.20 ((115, None),) ()


For most uses, `status`, `tie` and the targets are enough; the path is there when you must show why.

## 3.5 — Options of a conversion

`api.convert(...)` checks the arguments and reads the string, then calls the engine, `api.engine.convert(...)` (`idtrack.engine.Track.convert`). Use the API: it takes `allow_bridges`, `budget_states` and `budget_seconds`. The engine is strict where the API is forgiving: a `from_release` at which the identifier does not exist raises `idtrack.errors.InactiveAtReleaseError` (naming the releases where it exists), and an unknown version raises `idtrack.errors.UnknownVersionError`.

### 3.5.1 Bridges

The search follows Ensembl's history first. Only where that leads to no gene at the target release may it take a **bridge**: a passage from one gene to another through an identifier the two genes carried one after the other (an external id, or a transcript). A bridge is taken only where its checks say it is evidence that one gene became the other: never through a synonym, never through clone-based names, Rfam families or hyper-connected identifiers (3.5.2); where both genes lie on one assembly, only when their places overlap. A path takes at most two bridges (`idtrack.settings.SearchSettings.bridge_limit`), and fewer bridges rank first. `candidate.bridges` counts them, `result.path.bridges` (with `explain=True`) lists every bridge the search considered, taken or refused, with its rule, and `allow_bridges=False` answers by Ensembl's history alone. The full rules: [How idtrack answers](../concepts.rst).

Human `ENSG00000198477` (SUHW2 at release 48, ZNF280B later, retired after release 75) has no successor in Ensembl's history; a bridge through NCBI's id 140883 and RefSeq's NM_080764, at the same place on chromosome 22, reaches ZNF280B on GRCh38:

In [14]:
bridged = api.convert('ENSG00000198477', from_release=48, to_release=76, explain=True)
show(bridged)
for check in bridged.path.bridges:
    through = [f'{i} ({rule})' for i, rule in check.through if rule != 'synonym']
    print(f'  {check.source} -> {check.target}: places {check.places}, taken={check.taken}, through {through}')
show(api.convert('ENSG00000198477', from_release=48, to_release=76, allow_bridges=False))

'ENSG00000198477' read as 'ENSG00000198477' (match: exact) at release 48 -> ensembl_gene at 76: status=converted, tie=False, decided_by=None
  ENSG00000275004.1 -> ENSG00000275004.1  best=True entry=ENSG00000198477.2 bridges=1 score=0.932
  ENSG00000198477.2 -> ENSG00000198477.3: places same_place, taken=True, through ['140883 (same_place)', 'Q86YH2 (same_place)']
  ENSG00000198477.2 -> ENSG00000275004.1: places same_place, taken=True, through ['140883 (same_place)', 'NM_080764 (same_place)']
  ENSG00000198477.2 -> ENSG00000277421.1: places outside_reference, taken=False, through ['140883 (copy_id_elsewhere)']
  ENSG00000198477.3 -> ENSG00000168214.16: places places_apart, taken=False, through ['3516 (places_apart)']
  ENSG00000198477.3 -> ENSG00000168214.17: places places_apart, taken=False, through ['3516 (places_apart)']
  ENSG00000198477.3 -> ENSG00000275004.1: places same_place, taken=True, through ['140883 (same_place)', 'NM_080764 (same_place)', 'OTTHUMG00000151066 (same_place)'

The bridges between genes at the same place (`same_place`) were taken. Refused were those to a copy outside the reference sequence that no release gave the id together with the other gene (`copy_id_elsewhere`), and those to a gene at another place of the genome (`places_apart`, through NCBI's id 3516). The synonyms (left out of the print) are never a bridge. Every rule: `idtrack.results.BRIDGE_RULES`.

The search is bounded: each state is visited once, and a call that needs more than its budget (`idtrack.settings.SearchSettings`: 200,000 states and 30 s by default; `budget_states=` and `budget_seconds=` of a call) raises `idtrack.errors.SearchBudgetExceededError`.

### 3.5.2 Hyper-connected identifiers

Some external identifiers are carried by many genes at once (Rfam families, `U6`, `5S_rRNA`, snoRNA names): more than the hyper-connectivity threshold of the graph (`idtrack.settings.GraphSettings.hyperconnective_threshold`, 5) genes in one release. They name a kind of gene, not a gene, so they are never a bridge and are not counted as annotation of a gene. They can still be asked for and returned.

### 3.5.3 Assemblies

A conversion uses, at every release, the assembly Ensembl maintained at that release (human: NCBI36 up to release 54, GRCh37 up to 75, GRCh38 from 76). The GRCh37 of later releases is not walked; its ids are in the graph as ids to convert from. The final step into a database is made on the assembly of the target release, so a conversion to release 70 returns the HGNC symbols of GRCh37 at release 70.

So you can feed identifiers of older builds and bring them to one target release; an external id is returned as its database wrote it at the target release.

To read a string as an id of one database only, name it: `api.convert('TP53', from_database='HGNC Symbol')`.

In [15]:
# The hyper-connected identifiers of this graph, and bridges on or off for a gene that needs none
g = api.graph
hc = g.hyperconnective_nodes
print('Threshold (genes at once):', g.graph_settings.hyperconnective_threshold)
print('Hyper-connected external identifiers:', len(hc))
print('Top 10 by the number of genes they are linked to in one release:')
for node, genes in sorted(hc.items(), key=lambda kv: (-kv[1], kv[0]))[:10]:
    print(' ', genes, '-', node)

for allow_bridges in (False, True):
    res = api.convert('TP53', to_release=SNAPSHOT_RELEASE, allow_bridges=allow_bridges)
    print(f'TP53 allow_bridges={allow_bridges!s:<5} ->', {c.gene: c.bridges for c in res.view('best')})

Threshold (genes at once): 5
Hyper-connected external identifiers: 10881
Top 10 by the number of genes they are linked to in one release:
  1562 - RF00026
  1339 - U6
  1266 - RF00017
  1013 - RF00019
  955 - Y_RNA
  905 - Metazoa_SRP
  615 - RF00001
  527 - 5S_rRNA
  489 - SRP_euk_arch
  480 - RF01210
TP53 allow_bridges=False -> {'ENSG00000141510.20': 0}
TP53 allow_bridges=True  -> {'ENSG00000141510.20': 0}


## 3.6 — Questions to ask the graph

- Which external databases does it hold, and through which form (gene, transcript, translation)?
- Which genome assemblies, and which releases of each?
- When was an identifier active?
- Which genes does a name name, and which names does a gene carry?

In [16]:
described = api.describe()
print('Assemblies:', {a.assembly: (a.name, min(a.releases), max(a.releases)) for a in described.assemblies})
print('Releases:', api.releases()[0], 'to', api.releases()[-1])

forms = api.database_forms()
print('\nForm of some databases:', {name: forms[name] for name in ['CCDS', 'HGNC Symbol', 'RefSeq_mRNA', 'Uniprot/SWISSPROT']})

g = api.graph
example_gene = 'ENSG00000141510.20'  # TP53 at the snapshot release
print('\nVersions of ENSG00000141510:', len(g.versions_of('ENSG00000141510')), 'from',
      g.versions_of('ENSG00000141510')[0], 'to', g.versions_of('ENSG00000141510')[-1])
print('Active ranges of', example_gene, ':', g.active_ranges(example_gene), '(inf: still active at the snapshot release)')

print()
for match in api.genes_named('TP53')[-3:]:
    print('genes_named:', match.database, match.gene, match.releases[0], '-', match.releases[-1])
print('names_of:', [(m.database, m.name, m.synonym) for m in api.names_of(example_gene, SNAPSHOT_RELEASE)][:6])

2026-11-24 11:52:09 INFO:idtrack.graph.graph: Graph-wide table being calculated: assembly_releases


Assemblies: {38: ('GRCh38', 76, 115), 37: ('GRCh37', 55, 115), 36: ('NCBI36', 48, 54)}
Releases: 48 to 115

Form of some databases: {'CCDS': 'transcript', 'HGNC Symbol': 'gene', 'RefSeq_mRNA': 'transcript', 'Uniprot/SWISSPROT': 'translation'}

Versions of ENSG00000141510: 17 from ENSG00000141510.4 to ENSG00000141510.20
Active ranges of ENSG00000141510.20 : [[115, inf]] (inf: still active at the snapshot release)

genes_named: Vega gene ENSG00000141510.14 79 - 80
genes_named: Vega gene ENSG00000141510.15 81 - 82
genes_named: Vega gene ENSG00000141510.16 83 - 88
names_of: [('EntrezGene', '7157', False), ('HGNC Symbol', 'TP53', False), ('HGNC Symbol', 'LFS1', True), ('HGNC Symbol', 'P53', True), ('NCBI gene (formerly Entrezgene)', 'TP53', False), ('UniProtKB Gene Name', 'TP53', False)]


## 3.7 — Practical advice

1. Record the **snapshot release** of your graph and the target release in your analysis notes (`api.describe()` gives the snapshot release and the graph's `content_digest`).
2. If you changed the externals configuration (Part 6), share it with your results.
3. When an answer is ambiguous, do not hide it: decide how your pipeline handles ties and 1→n answers, and record it.
4. For single-cell harmonization, convert Ensembl ids where you have them before symbols: they carry their history, a symbol does not.
5. Read [Limitations and known costs](../limitations.rst) once: it says where an answer can be wrong, with how often that was measured.

> **Tip:** Troubleshooting, diagnostics and pipelines: Part 7 (`07_advanced_topics.ipynb`).